# 04 — RAG Retrieval Quality
Validates the complete RAG pipeline: chunk building → embedding → Qdrant search
→ HyDE expansion → BM25 rerank → context assembly.

**Real-time integration**: RAG retrieval is on the WebSocket hot path.
This notebook measures:
- Embedding cache hit rates (target: > 80% for repeated column names)
- End-to-end retrieval latency (target: < 100ms embed + search + rerank)
- HyDE expansion quality improvement vs direct query embedding
- Context string relevance for a set of test queries


In [ ]:
import sys, time, asyncio
sys.path.insert(0, '..')
import polars as pl

retail = pl.read_csv('datasets/sample_retail_sales.csv')
hr     = pl.read_excel('datasets/sample_hr_data.xlsx', sheet_name='Employees')
print("Datasets loaded.")


## Chunk building — what gets indexed into Qdrant

In [ ]:
from backend.agents.data.rag.chunk_builder import ChunkBuilder

builder = ChunkBuilder()

# Simulate the schema dict that SchemaAgent produces
retail_schema = {
    'columns': [
        {'name': 'order_id',     'data_type': 'Utf8',    'semantic_type': 'identifier',     'null_rate': 0.0,  'unique_count': 500,  'sample_values': ['ORD-00001','ORD-00002']},
        {'name': 'revenue',      'data_type': 'Float64',  'semantic_type': 'currency',        'null_rate': 0.08, 'unique_count': 491,  'sample_values': ['1245.50','890.20']},
        {'name': 'region',       'data_type': 'Utf8',    'semantic_type': 'categorical',     'null_rate': 0.0,  'unique_count': 5,    'sample_values': ['North','South','East']},
        {'name': 'discount_pct', 'data_type': 'Float64',  'semantic_type': 'percentage',      'null_rate': 0.0,  'unique_count': 498,  'sample_values': ['0.05','0.15','0.22']},
        {'name': 'order_date',   'data_type': 'Utf8',    'semantic_type': 'date',            'null_rate': 0.0,  'unique_count': 340,  'sample_values': ['2024-01-15','2024-03-22']},
        {'name': 'status',       'data_type': 'Utf8',    'semantic_type': 'categorical',     'null_rate': 0.0,  'unique_count': 4,    'sample_values': ['completed','pending']},
        {'name': 'quantity',     'data_type': 'Int64',    'semantic_type': 'numeric_count',   'null_rate': 0.0,  'unique_count': 19,   'sample_values': ['3','7','12']},
        {'name': 'unit_price',   'data_type': 'Float64',  'semantic_type': 'currency',        'null_rate': 0.0,  'unique_count': 498,  'sample_values': ['45.00','72.50']},
        {'name': 'category',     'data_type': 'Utf8',    'semantic_type': 'categorical',     'null_rate': 0.0,  'unique_count': 5,    'sample_values': ['Electronics','Food']},
        {'name': 'product_sku',  'data_type': 'Utf8',    'semantic_type': 'identifier',      'null_rate': 0.0,  'unique_count': 498,  'sample_values': ['SKU-4521','SKU-7823']},
    ],
    'row_count_sample': 500,
    'column_count': 10,
}

schema_chunks = builder.build_schema_chunks('test_retail', retail_schema)
print(f"Schema chunks built: {len(schema_chunks)}")
for chunk in schema_chunks[:3]:
    print(f"\n── {chunk.column_name} ({chunk.chunk_type}) ──")
    print(chunk.content)


## HyDE Expansion — query quality improvement

In [ ]:
from backend.agents.data.rag.hyde_expander import HyDEExpander

# Mock LLM that returns plausible hypothetical answers
class MockLLM:
    RESPONSES = {
        'revenue': 'The revenue column shows total sales values ranging from $45 to $2,800 with a mean of $1,205. The North region generates 28% of total revenue at $142,000 annually.',
        'discount': 'The discount_pct column contains promotional discount rates between 0% and 30%. The average discount is 15.2%, with Electronics category receiving the highest average discount of 22%.',
        'anomaly': 'The revenue column has 3 negative values (-$120.50, -$45.00, -$890.00) which are likely data entry errors or refund records incorrectly signed.',
    }
    async def complete(self, prompt, **kwargs):
        for key, resp in self.RESPONSES.items():
            if key in prompt.lower():
                return resp
        return 'The dataset contains various business metrics including revenue, orders, and customer segments.'

expander = HyDEExpander(llm_client=MockLLM())

test_queries = [
    'what is the average revenue by region?',
    'what discount rates are applied to orders?',
    'are there any anomalies in the revenue column?',
    'how many orders are in pending status?',
]

schema_summary = 'Columns: order_id, revenue, region, category, quantity, unit_price, discount_pct, status, order_date, product_sku'

print("Query → HyDE Expanded Query\n")
for query in test_queries:
    expanded = asyncio.run(expander.expand(query, schema_summary))
    print(f"Original:  {query}")
    print(f"Expanded:  {expanded[:120]}...")
    print(f"Same:      {query == expanded}\n")


## Retrieval simulation — context relevance scoring

In [ ]:
# Simulate retrieval without a real Qdrant instance
# by scoring chunks against queries using simple keyword overlap (BM25-lite)

def bm25_score(chunk_content: str, query: str, k1=1.5, b=0.75) -> float:
    """Simplified BM25 scoring for retrieval quality estimation."""
    import math
    query_terms = set(query.lower().split())
    doc_terms   = chunk_content.lower().split()
    doc_len     = len(doc_terms)
    avg_doc_len = 50  # approximate average chunk length
    score = 0.0
    for term in query_terms:
        tf = doc_terms.count(term)
        if tf == 0:
            continue
        idf = math.log(1 + (len(schema_chunks) - 1 + 0.5) / (1 + 1))
        tf_norm = tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len / avg_doc_len))
        score += idf * tf_norm
    return round(score, 4)

test_queries_eval = [
    ('what is the total revenue?',           'revenue'),
    ('which region has most orders?',        'region'),
    ('what discount rates are used?',        'discount_pct'),
    ('show me the order status breakdown',   'status'),
]

print("Retrieval Quality — Top-3 chunks per query\n")
for query, expected_col in test_queries_eval:
    scored = sorted(
        [(chunk, bm25_score(chunk.content, query)) for chunk in schema_chunks],
        key=lambda x: -x[1]
    )
    top3_cols = [c.column_name for c, _ in scored[:3]]
    hit = expected_col in top3_cols
    print(f"Query:    {query}")
    print(f"Expected: {expected_col} | Top-3: {top3_cols} | Hit: {'✓' if hit else '✗'}")
    print(f"Scores:   {[(c.column_name, s) for c, s in scored[:3]]}\n")


## Context string assembly — token budget check

In [ ]:
# Verify that the assembled context string fits within the LLM context window
# The RAGAgent uses max_chars=3000 to stay within the system prompt budget.

from backend.agents.data.rag.retriever import Retriever

# Mock results in the format Retriever.build_context_string() expects
mock_results = [
    {
        'score': 0.85,
        'payload': {'chunk_type': 'column_description', 'content': chunk.content}
    }
    for chunk in schema_chunks[:8]
]

retriever = Retriever.__new__(Retriever)  # skip __init__ to avoid Qdrant connection
context   = retriever.build_context_string(mock_results, max_chars=3000)

# Rough token estimate: ~4 chars per token
estimated_tokens = len(context) / 4
print(f"Context string length:    {len(context)} chars")
print(f"Estimated tokens:         {estimated_tokens:.0f}")
print(f"Fits in system prompt:    {'✓' if estimated_tokens < 800 else '✗'} (< 800 token budget)")
print(f"\nContext preview (first 400 chars):\n{context[:400]}...")


## Embedding cache hit rate simulation

In [ ]:
# Simulate the embedding cache to estimate hit rate in production.
# Column names are reused across many user queries — high cache hit rate
# is the primary reason embed latency < 80ms on the hot path.

import hashlib, collections

# Simulate 100 chat queries asking about the retail dataset
query_templates = [
    'what is the total {col}?',
    'show me {col} by region',
    'what is the average {col}?',
    'find anomalies in {col}',
    'plot {col} over time',
]
cols = ['revenue', 'quantity', 'discount_pct', 'unit_price', 'status']

queries = []
for _ in range(100):
    import random; random.seed(_)
    col = random.choice(cols)
    tmpl = random.choice(query_templates)
    queries.append(tmpl.format(col=col))

cache_keys = [hashlib.sha256(q.encode()).hexdigest()[:8] for q in queries]
cache       = {}
hits = 0
for key, query in zip(cache_keys, queries):
    if key in cache:
        hits += 1
    else:
        cache[key] = f'embedding_for_{query[:20]}'

print(f"Simulated {len(queries)} chat queries against retail dataset")
print(f"Cache hits:  {hits} / {len(queries)} = {hits/len(queries):.0%}")
print(f"Cache misses:{len(queries)-hits} (each costs ~80ms Bedrock call)")
print(f"Total embed time without cache: {len(queries)*80}ms")
print(f"Total embed time with cache:    {(len(queries)-hits)*80}ms")
